# Route-Wise Traffic Accident Risk Prediction and Safer Travel Time Recommendation System

**Course:** 213CSE2301 - Predictive Analytics (Autonomy)  
**Course Coordinator:** Dr. N. Subbulakshmi  
**Department:** Department of Computer Science and Engineering  

### Team Members:
1. **99240040465** - E Goutham Reddy
2. **99240040477** - D Karimulla
3. **99240040480** - G S N Abhiram
4. **99240040466** - U Sasi Kumar
5. **99240040467** - E Manideep

---
## Project Objectives & Architecture
1. **Aggregated Multi-Source Dataset:** 24,000 raw incidents cleaned down to **6,469 verified records** across 5 datasets.
2. **Accident Blackspot Detection:** K-Means Spatial Clustering (20 clusters) computing baseline spatial risk scores.
3. **Predictive Modeling:** Multi-class classification (Random Forest, Logistic Regression, Decision Tree / ID3, k-NN, Naive Bayes, SVM) predicting severity (, , ).
4. **Route Risk Scoring:** Multi-segment risk scoring along any selected origin-destination road route.
5. **Safer Travel Time Recommender:** Evaluating route risks across temporal windows (*Morning*, *Afternoon*, *Evening*, *Night*) to recommend the safest departure time with quantified risk reduction.

In [ ]:
# 1. Imports and Environment Setup
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium import plugins
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from IPython.display import display
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

sns.set_theme(style="darkgrid")
plt.rcParams["figure.dpi"] = 120
print("Environment initialized successfully.")

## 2. Data Loading & Ingestion Pipeline
Loads the verified reference dataset containing 6,469 cleaned records (or generates it via  if not present).

In [ ]:
dataset_path = "data/cleaned_accidents_6469.csv"
if not os.path.exists(dataset_path):
    from src.data_pipeline import run_pipeline
    run_pipeline("data")

df = pd.read_csv(dataset_path)
print(f"Dataset successfully loaded. Shape: {df.shape}")
df.head()

## 3. Exploratory Data Analysis & Visualizations
Replication of the project reference presentation charts (Slides 5, 6, and 9).

In [ ]:
# A. Accident Frequency by Time Category (Slide 5)
plt.figure(figsize=(9, 4))
time_order = ["Morning", "Afternoon", "Evening", "Night"]
colors_time = ["#434574", "#326f81", "#2ea07d", "#72b855"]
ax = sns.countplot(data=df, x="Time_Category", order=time_order, palette=colors_time)
plt.title("Accident Frequency by Time Category (Slide 5)", fontsize=13)
plt.xlabel("Time of Day")
plt.ylabel("Total Accidents")
for p in ax.patches:
    h = int(p.get_height())
    ax.annotate(f"{h:,}", (p.get_x() + p.get_width()/2., h), ha="center", va="bottom", fontsize=9)
plt.show()

# B. Severity Partitioned by Time Category (Slide 6)
plt.figure(figsize=(10, 5))
palette_sev = {"High": "#552147", "Medium": "#b22c54", "Low": "#e0886b"}
ax = sns.countplot(data=df, x="Time_Category", hue="Severity", order=time_order, hue_order=["High", "Medium", "Low"], palette=palette_sev)
plt.title("Accident Severity Partitioned by Time Category (Slide 6)", fontsize=13)
plt.xlabel("Time Segment")
plt.ylabel("Count")
plt.legend(title="Severity")
plt.show()

# C. 24-Hour Histogram and Correlation Heatmap (Slide 9)
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
hours = df["Hour"].value_counts().sort_index()
axes[0].bar(hours.index, hours.values, color="#6b96db", edgecolor="black", width=0.8)
axes[0].set_title("Accident Frequency by Hour of Day (0-23)")
axes[0].set_xlabel("Hour")
axes[0].set_ylabel("Count")
axes[0].set_xticks(range(24))

le_t = LabelEncoder()
le_s = LabelEncoder()
df_corr = pd.DataFrame({
    "Hour": df["Hour"],
    "Time_Encoded": le_t.fit_transform(df["Time_Category"]),
    "Target_Encoded": le_s.fit_transform(df["Severity"])
})
sns.heatmap(df_corr.corr(), annot=True, fmt=".3f", cmap="RdBu_r", vmin=-1.0, vmax=1.0, ax=axes[1])
axes[1].set_title("Pearson Correlation Heatmap (Slide 9)")
plt.tight_layout()
plt.show()

## 4. Accident-Prone Zone Detection via K-Means Clustering
We group coordinates into 20 geographical clusters to detect accident blackspots and derive baseline  values.

In [ ]:
NUM_CLUSTERS = 20
kmeans = KMeans(n_clusters=NUM_CLUSTERS, random_state=42, n_init=10)
df["Cluster"] = kmeans.fit_predict(df[["Latitude", "Longitude"]])

sev_score_map = {"Low": 1.0, "Medium": 2.0, "High": 3.0}
df["Severity_Numeric"] = df["Severity"].map(sev_score_map)
cluster_risk = df.groupby("Cluster")["Severity_Numeric"].mean().reset_index()
cluster_risk.rename(columns={"Severity_Numeric": "Cluster_Risk_Score"}, inplace=True)
df = df.merge(cluster_risk, on="Cluster", how="left")

print("Accident Hotspots (K-Means Clusters) detected successfully:")
df[["Latitude", "Longitude", "Cluster", "Cluster_Risk_Score", "Severity"]].head()

## 5. Machine Learning Modeling & Lab Classifier Comparison
We train the Random Forest Classifier alongside all comparative algorithms studied in the curriculum:
- Logistic Regression
- Decision Tree (ID3)
- k-Nearest Neighbors (k-NN)
- Naive Bayes (GaussianNB)
- Support Vector Machine (SVM)

In [ ]:
le_time = LabelEncoder()
df["Time_Cat_Encoded"] = le_time.fit_transform(df["Time_Category"])

le_weather = LabelEncoder()
df["Weather_Cat_Encoded"] = le_weather.fit_transform(df["Weather"])

le_road = LabelEncoder()
df["Road_Cat_Encoded"] = le_road.fit_transform(df["Road_Type"])

le_target = LabelEncoder()
df["Target_Encoded"] = le_target.fit_transform(df["Severity"])

feature_cols = ["Hour", "Time_Cat_Encoded", "Weather_Cat_Encoded", "Road_Cat_Encoded", "Cluster_Risk_Score"]
X = df[feature_cols]
y = df["Target_Encoded"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

models = {
    "Random Forest": RandomForestClassifier(n_estimators=120, max_depth=9, random_state=42, class_weight="balanced"),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree (ID3)": DecisionTreeClassifier(max_depth=8, criterion="entropy", random_state=42),
    "k-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes (Gaussian)": GaussianNB(),
    "Support Vector Machine": SVC(kernel="rbf", random_state=42)
}

results = []
for name, m in models.items():
    m.fit(X_train, y_train)
    y_pred = m.predict(X_test)
    results.append({
        "Algorithm": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, average="weighted", zero_division=0)
    })

comp_df = pd.DataFrame(results)
print("=== MODEL PERFORMANCE COMPARISON (Course 213CSE2301) ===")
display(comp_df)

### Model Evaluation & Confusion Matrix (Slide 7)

In [ ]:
rf_model = models["Random Forest"]
y_pred_rf = rf_model.predict(X_test)
classes = list(le_target.classes_)

cm = confusion_matrix(y_test, y_pred_rf)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes)
plt.title("Confusion Matrix - Random Forest", fontsize=13)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

# Feature Importance
plt.figure(figsize=(7, 3))
sns.barplot(x=rf_model.feature_importances_, y=feature_cols, palette="coolwarm")
plt.title("Feature Importance - Random Forest (Slide 8)")
plt.xlabel("Gini Importance Score")
plt.show()

## 6. Route Risk Engine & Safer Travel Time Recommendation
Helper functions to calculate route risk across road waypoints and compare travel times.

In [ ]:
from src.models import AccidentRiskModel
from src.router import get_coordinates, RouteRiskAnalyzer
from src.visualizer import build_interactive_route_map

model_engine = AccidentRiskModel()
model_engine.load_artifacts("models")
analyzer = RouteRiskAnalyzer(model_engine)

origin = "Connaught Place Delhi"
destination = "India Gate Delhi"
departure_hour = 23  # Late Night
time_category = "Night"
weather = "Clear"
road_type = "Arterial"

c_start = get_coordinates(origin)
c_end = get_coordinates(destination)

route_eval = analyzer.analyze_route(c_start, c_end, departure_hour, time_category, weather, road_type)
rec_eval = analyzer.recommend_safest_departure_time(c_start, c_end, departure_hour, time_category, weather, road_type)

print("=" * 55)
print("JOURNEY RISK ANALYSIS REPORT")
print("=" * 55)
print(f"Route:                  {origin} -> {destination}")
print(f"Distance:               {route_eval['distance_km']} km")
print(f"Est. Travel Duration:   {route_eval['estimated_minutes']} mins")
print(f"Current Departure:      {time_category} ({departure_hour:02d}:00 hrs)")
print(f"Predicted Journey Risk: {route_eval['journey_risk_tier']} (Score: {route_eval['mean_risk_score']:.3f})")
print("-" * 55)
print(f"SAFEST TRAVEL WINDOW:   {rec_eval['safest_time_category']}")
print(f"QUANTIFIED RISK REDUCTION: {rec_eval['risk_reduction_pct']}% safer than current departure")
print("=" * 55)

## 7. Interactive Route Map Visualization
Interactive map rendering the evaluated route path, origin & destination markers, and historical accident blackspot clusters.

In [ ]:
m = build_interactive_route_map(
    c_start, c_end, route_eval, model_engine,
    time_cat=time_category, weather=weather, road_type=road_type,
    origin_name=origin, dest_name=destination
)
m